# Several Series at Once

Download a handful of ECB series and align them on `TIME_PERIOD` into one panel. We use a small helper so each fetch is one line.

In [ ]:
Pollis.packages("HTTP", "CSV", "Tables", "TimeSeries")

In [ ]:
using HTTP, CSV, Tables

base = "https://data-api.ecb.europa.eu/service/data"

# Fetch any series as a Tables.jl table
function ecb(flow, key; params = "format=csvdata")
    url = "$base/$flow/$key?$params"
    return CSV.File(HTTP.get(url).body)
end

## Three Euro Reference Rates

Use the `+` operator to request several currencies in one call, or fetch them separately.

In [ ]:
p = "startPeriod=2022-01-01&format=csvdata"

usd = ecb("EXR", "D.USD.EUR.SP00.A"; params = p)
gbp = ecb("EXR", "D.GBP.EUR.SP00.A"; params = p)
jpy = ecb("EXR", "D.JPY.EUR.SP00.A"; params = p)

println("USD rows: ", length(Tables.getcolumn(usd, :OBS_VALUE)))

## Align on Date

Build a lookup from date to value for each series, then join on the shared dates. This stays in plain Julia — no frame library required.

In [ ]:
asdict(t) = Dict(zip(Tables.getcolumn(t, :TIME_PERIOD), Tables.getcolumn(t, :OBS_VALUE)))

du, dg, dj = asdict(usd), asdict(gbp), asdict(jpy)
common = sort(collect(intersect(keys(du), keys(dg), keys(dj))))

println("Aligned dates: ", length(common))
for d in last(common, 3)
    println(d, "  USD=", du[d], "  GBP=", dg[d], "  JPY=", dj[d])
end

## Or Merge into a TimeArray

With **TimeSeries.jl**, wrap each series as a `TimeArray` and let `merge` align them on the timestamp (an inner join on date) — no DataFrames needed.

In [ ]:
using TimeSeries, Dates

# EXR is daily; parse TIME_PERIOD to Date and pair with OBS_VALUE
ta(t, name) = TimeArray(Date.(string.(Tables.getcolumn(t, :TIME_PERIOD))),
                        Tables.getcolumn(t, :OBS_VALUE), [name])

panel = merge(ta(usd, :USD), ta(gbp, :GBP), ta(jpy, :JPY))   # inner join on date
last(panel, 5)

## Summary
- A small `ecb(flow, key)` helper keeps each fetch to one line.
- Align series on the date either in plain Julia (dicts + `intersect`) or by merging `TimeArray`s with TimeSeries.jl.
- The `+` operator (`D.USD+GBP+JPY.EUR.SP00.A`) can also return several currencies in one request.